# bb.ipynb — 이륜차/헬멧/싸움/스토킹/침입 통합 이상행동 파이프라인

이 노트북은 침입 파트처럼 **단계별 산출물 JSON을 남기고**, 마지막에 여러 후보 이벤트를 병합하는 구조입니다.

구성:
1. `base_tracking`: 사람 + 이륜 이동수단 추적
2. `scene_detection`: YOLO-World로 헬멧, 인도/도로, 이륜차, 장면 구조 후보 검출
3. `sidewalk_segmentation`: SegFormer Cityscapes가 가능하면 도로/인도 픽셀 추론, 불가능하면 YOLO-World bbox 기반 fallback
4. `helmet_reasoning`: rider-person association + head-zone helmet/bare-head 근거
5. `sidewalk_reasoning`: 이륜차 기준점이 인도 영역에 머무는지 판단
6. `fight_reasoning`: 사람 간 근접도 + 움직임/optical flow 기반 싸움 후보
7. `stalking_reasoning`: 두 사람 track의 지속 거리/동행 방향/뒤따름 관계 기반 스토킹 후보
8. `intrusion_optional`: 업로드된 침입 파이프라인 파일이 있으면 선택적으로 함께 실행
9. `fusion`: 후보군 중 가장 강한 이벤트를 final로 선택하고 전체 후보를 JSON으로 저장

> 실행 전: 이 노트북과 `yolo11s.pt`, `yolo11s-pose.pt`, `yolov8s-world.pt`를 같은 폴더에 두는 것을 권장합니다. 영상은 `Sample`, `sample`, `Sample (1)/Sample` 중 하나에 있으면 자동 탐색합니다.

In [1]:
# ============================================================
# 0. Imports / 기본 설정
# ============================================================
from pathlib import Path
from collections import defaultdict, Counter
import os
import sys
import json
import math
import time
import traceback

import cv2
import numpy as np

try:
    from ultralytics import YOLO
except Exception as e:
    YOLO = None
    print('[warn] ultralytics import 실패:', e)

try:
    import torch
except Exception:
    torch = None

# 현재 노트북 실행 위치를 프로젝트 루트 후보로 사용
BASE_DIR = Path.cwd()
HOME_DIR = Path.home()

# ------------------------------------------------------------
# 영상 폴더 설정
# ------------------------------------------------------------
# 자동 탐색이 실패하면 아래 USER_SAMPLE_DIR 한 줄만 네 실제 경로로 바꾸고
# 이 셀부터 다시 실행하면 된다.
# 예: USER_SAMPLE_DIR = Path(r'C:/Users/우성한/Sample (1)/Sample')
USER_SAMPLE_DIR = None
VIDEO_EXTS = ('.mp4', '.avi', '.mov', '.mkv', '.wmv', '.MP4', '.AVI', '.MOV', '.MKV', '.WMV')


def _as_path_or_none(x):
    if x is None:
        return None
    s = str(x).strip()
    if not s:
        return None
    return Path(s).expanduser()


def _dedupe_existing_dirs(paths):
    out = []
    seen = set()
    for p in paths:
        p = _as_path_or_none(p)
        if p is None:
            continue
        if p.is_file():
            p = p.parent
        if not p.exists() or not p.is_dir():
            continue
        try:
            key = str(p.resolve()).lower()
        except Exception:
            key = str(p).lower()
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out


def _candidate_video_dirs(user_dir=None):
    env_dirs = [
        os.environ.get('VIDEO_ROOT'),
        os.environ.get('SAMPLE_DIR'),
        os.environ.get('DATASET_DIR'),
    ]
    roots = [
        user_dir,
        *env_dirs,
        BASE_DIR,
        BASE_DIR / 'Sample',
        BASE_DIR / 'sample',
        BASE_DIR / 'videos',
        BASE_DIR / 'video',
        BASE_DIR / 'Sample (1)',
        BASE_DIR / 'Sample (1)' / 'Sample',
        BASE_DIR.parent,
        BASE_DIR.parent / 'Sample',
        BASE_DIR.parent / 'sample',
        BASE_DIR.parent / 'videos',
        BASE_DIR.parent / 'Sample (1)',
        BASE_DIR.parent / 'Sample (1)' / 'Sample',
        HOME_DIR / 'Sample',
        HOME_DIR / 'sample',
        HOME_DIR / 'videos',
        HOME_DIR / 'Videos',
        HOME_DIR / 'Desktop' / 'Sample',
        HOME_DIR / 'Documents' / 'Sample',
        HOME_DIR / 'Downloads' / 'Sample',
        HOME_DIR / 'Sample (1)',
        HOME_DIR / 'Sample (1)' / 'Sample',
    ]
    return _dedupe_existing_dirs(roots)


def _dir_has_video(path: Path, recursive=False):
    try:
        iterator = path.rglob('*') if recursive else path.iterdir()
        for p in iterator:
            if p.is_file() and p.suffix in VIDEO_EXTS:
                return True
    except Exception:
        pass
    return False


def _scan_video_dirs_limited(roots=None, max_depth=4, max_found=20):
    """
    자동 후보에서 영상을 못 찾았을 때만 제한적으로 폴더를 훑는다.
    너무 넓게 스캔하지 않도록 depth와 found 수를 제한한다.
    """
    if roots is None:
        roots = [BASE_DIR, BASE_DIR.parent, HOME_DIR, HOME_DIR / 'Sample (1)']
    found = []
    seen = set()

    def walk(root: Path, depth: int):
        if len(found) >= max_found:
            return
        root = _as_path_or_none(root)
        if root is None or not root.exists() or not root.is_dir():
            return
        try:
            key = str(root.resolve()).lower()
        except Exception:
            key = str(root).lower()
        if key in seen:
            return
        seen.add(key)

        if _dir_has_video(root, recursive=False):
            found.append(root)
            return
        if depth >= max_depth:
            return

        try:
            children = list(root.iterdir())
        except Exception:
            return

        # Sample/video 이름을 우선 탐색
        children = sorted(
            [c for c in children if c.is_dir()],
            key=lambda p: (0 if any(k in p.name.lower() for k in ['sample', 'video', 'dataset']) else 1, p.name.lower())
        )
        for child in children:
            # 너무 큰 시스템성 폴더는 제외
            if child.name.lower() in {'.git', '__pycache__', 'appdata', 'node_modules', '.cache'}:
                continue
            walk(child, depth + 1)
            if len(found) >= max_found:
                break

    for r in roots:
        walk(r, 0)
    return _dedupe_existing_dirs(found)


def resolve_video_search_dirs(user_dir=None):
    dirs = _candidate_video_dirs(user_dir)

    # 우선 후보 중 실제 영상이 들어 있는 폴더를 앞으로 보낸다.
    with_video = [d for d in dirs if _dir_has_video(d, recursive=True)]
    without_video = [d for d in dirs if d not in with_video]

    # 후보에서 못 찾으면 제한 스캔
    if not with_video:
        scanned = _scan_video_dirs_limited()
        with_video = [d for d in scanned if _dir_has_video(d, recursive=True)]

    return _dedupe_existing_dirs(with_video + without_video)


VIDEO_SEARCH_DIRS = resolve_video_search_dirs(USER_SAMPLE_DIR)
SAMPLE_DIR = VIDEO_SEARCH_DIRS[0] if VIDEO_SEARCH_DIRS else (Path(USER_SAMPLE_DIR) if USER_SAMPLE_DIR else BASE_DIR / 'Sample')

# 출력 폴더
MODEL_DIR = BASE_DIR / 'models'
OUTPUT_DIR = BASE_DIR / 'outputs' / 'bb'
TRACKING_DIR = OUTPUT_DIR / 'tracking'
SCENE_DIR = OUTPUT_DIR / 'scene'
SEG_DIR = OUTPUT_DIR / 'segmentation'
PRED_DIR = OUTPUT_DIR / 'prediction'
FINAL_DIR = OUTPUT_DIR / 'final'
DEBUG_DIR = OUTPUT_DIR / 'debug'

for d in [OUTPUT_DIR, TRACKING_DIR, SCENE_DIR, SEG_DIR, PRED_DIR, FINAL_DIR, DEBUG_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def first_existing(*paths, default=None):
    for p in paths:
        if p is None:
            continue
        p = Path(p)
        if p.exists():
            return p
    return default

PERSON_MODEL = first_existing(
    BASE_DIR / 'yolo11s.pt',
    MODEL_DIR / 'yolo11s.pt',
    Path('/mnt/data/yolo11s.pt'),
    default='yolo11s.pt',
)

POSE_MODEL = first_existing(
    BASE_DIR / 'yolo11s-pose.pt',
    MODEL_DIR / 'yolo11s-pose.pt',
    Path('/mnt/data/yolo11s-pose.pt'),
    default='yolo11s-pose.pt',
)

YOLO_WORLD_MODEL = first_existing(
    BASE_DIR / 'yolov8s-world.pt',
    MODEL_DIR / 'yolov8s-world.pt',
    Path('/mnt/data/yolov8s-world.pt'),
    default='yolov8s-world.pt',
)

# 속도/정확도 조절 기본값
IMG_SIZE = 1280
TRACK_CONF = 0.12
WORLD_CONF = 0.035
POSE_CONF = 0.12
TRACKER_YAML = 'bytetrack.yaml'  # 'botsort.yaml'도 가능

# 실행 스위치
USE_SEGFORMER_FOR_SIDEWALK = True     # transformers/torch 사용 가능할 때만 동작
RUN_INTRUSION_BY_DEFAULT = False      # 침입까지 함께 돌리려면 run_bb_pipeline(..., run_intrusion=True)

# device 자동 설정
if torch is not None and torch.cuda.is_available():
    DEVICE = 0
    print('[device] GPU:', torch.cuda.get_device_name(0))
else:
    DEVICE = 'cpu'
    print('[device] CPU')

print('[cwd]', BASE_DIR)
print('[sample_dir]', SAMPLE_DIR)
print('[video_search_dirs]', [str(p) for p in VIDEO_SEARCH_DIRS[:8]])

# COCO class id
COCO_PERSON = 0
COCO_BICYCLE = 1
COCO_MOTORCYCLE = 3
TWO_WHEEL_COCO_IDS = {COCO_BICYCLE, COCO_MOTORCYCLE}
PERSON_AND_TWO_WHEEL_IDS = [COCO_PERSON, COCO_BICYCLE, COCO_MOTORCYCLE]

TWO_WHEEL_LABELS = {
    'bicycle', 'bike', 'motorcycle', 'motorbike', 'scooter', 'electric scooter',
    'kickboard', 'e-scooter', 'moped', 'two wheeler', 'rider'
}

HELMET_LABELS = {
    'helmet', 'motorcycle helmet', 'safety helmet', 'bike helmet'
}

BARE_HEAD_LABELS = {
    'bare head', 'head without helmet', 'person without helmet', 'no helmet', 'head'
}

SIDEWALK_LABELS = {'sidewalk', 'pavement', 'pedestrian path', 'walkway'}
ROAD_LABELS = {'road', 'street', 'lane', 'driveway'}

SCENE_PROMPTS = [
    # two-wheeler / rider
    'bicycle', 'motorcycle', 'scooter', 'electric scooter', 'kickboard', 'moped',
    'rider',
    # helmet
    'helmet', 'motorcycle helmet', 'safety helmet', 'bike helmet',
    'bare head', 'head without helmet', 'person without helmet', 'no helmet', 'head',
    # road / sidewalk
    'sidewalk', 'pavement', 'pedestrian path', 'walkway', 'road', 'street', 'lane',
    # intrusion scene structures optional
    'gate', 'entrance gate', 'fence', 'railing', 'barrier', 'wall', 'low wall',
]


[device] GPU: NVIDIA GeForce RTX 3060
[cwd] C:\Users\우성한
[sample_dir] C:\Users\우성한
[video_search_dirs] ['C:\\Users\\우성한', 'C:\\Users\\우성한\\Sample (1)', 'C:\\Users\\우성한\\Sample (1)\\Sample', 'C:\\Users', 'C:\\Users\\우성한\\videos']


In [2]:
# ============================================================
# 1. 공통 유틸
# ============================================================
def safe_stem(name: str) -> str:
    return Path(str(name)).stem.replace('(', '_').replace(')', '').replace(' ', '_')


def save_json(data, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print('[save]', path)


def load_json(path: Path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'JSON 파일이 없습니다: {path}')
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def set_video_root(path):
    """
    영상 폴더를 수동 지정한다.
    예: set_video_root(r'C:/Users/우성한/Sample (1)/Sample')
    """
    global USER_SAMPLE_DIR, SAMPLE_DIR, VIDEO_SEARCH_DIRS
    USER_SAMPLE_DIR = Path(path).expanduser()
    VIDEO_SEARCH_DIRS = resolve_video_search_dirs(USER_SAMPLE_DIR)
    # 수동 지정 경로는 영상이 없어도 가장 앞에 유지해서 오류 메시지를 명확히 한다.
    if USER_SAMPLE_DIR.exists() and USER_SAMPLE_DIR not in VIDEO_SEARCH_DIRS:
        VIDEO_SEARCH_DIRS = [USER_SAMPLE_DIR] + VIDEO_SEARCH_DIRS
    SAMPLE_DIR = VIDEO_SEARCH_DIRS[0] if VIDEO_SEARCH_DIRS else USER_SAMPLE_DIR
    print('[set_video_root]', SAMPLE_DIR)
    print_video_diagnostics(max_show=10)
    return SAMPLE_DIR


def refresh_video_dirs():
    global SAMPLE_DIR, VIDEO_SEARCH_DIRS
    VIDEO_SEARCH_DIRS = resolve_video_search_dirs(USER_SAMPLE_DIR)
    SAMPLE_DIR = VIDEO_SEARCH_DIRS[0] if VIDEO_SEARCH_DIRS else (Path(USER_SAMPLE_DIR) if USER_SAMPLE_DIR else BASE_DIR / 'Sample')
    print('[refresh] sample_dir:', SAMPLE_DIR)
    return VIDEO_SEARCH_DIRS


def list_videos(sample_dir: Path = None, exts=VIDEO_EXTS):
    if sample_dir is not None:
        search_dirs = _dedupe_existing_dirs([sample_dir])
    else:
        search_dirs = VIDEO_SEARCH_DIRS or refresh_video_dirs()

    videos = []
    seen = set()
    for root in search_dirs:
        root = Path(root)
        if not root.exists():
            continue
        try:
            for p in root.rglob('*'):
                if p.is_file() and p.suffix in exts:
                    try:
                        key = str(p.resolve()).lower()
                    except Exception:
                        key = str(p).lower()
                    if key not in seen:
                        seen.add(key)
                        videos.append(p)
        except Exception as e:
            print('[warn] 영상 탐색 실패:', root, e)
    return sorted(videos, key=lambda p: str(p).lower())


def find_video(video_name, sample_dir: Path = None) -> Path:
    p = Path(str(video_name)).expanduser()
    if p.exists() and p.is_file():
        return p

    search_dirs = _dedupe_existing_dirs([sample_dir]) if sample_dir is not None else (VIDEO_SEARCH_DIRS or refresh_video_dirs())
    target = str(video_name)
    target_name = Path(target).name
    target_stem = Path(target).stem

    # 1) 파일명 완전 일치
    for root in search_dirs:
        try:
            matches = list(Path(root).rglob(target_name))
        except Exception:
            matches = []
        if matches:
            return matches[0]

    # 2) stem 포함 검색
    for v in list_videos(sample_dir):
        if v.stem == target_stem or target_stem in v.name or target_name in v.name:
            return v

    msg = [
        f'영상 파일을 찾지 못했습니다: {video_name}',
        f'cwd={BASE_DIR}',
        f'SAMPLE_DIR={SAMPLE_DIR}',
        'VIDEO_SEARCH_DIRS=' + json.dumps([str(p) for p in search_dirs[:12]], ensure_ascii=False),
        '해결: set_video_root(r"C:/Users/우성한/Sample (1)/Sample") 실행 후 다시 시도하세요.'
    ]
    raise RuntimeError('\n'.join(msg))


def print_video_diagnostics(max_show=20):
    videos = list_videos()
    print('cwd:', BASE_DIR)
    print('SAMPLE_DIR:', SAMPLE_DIR, '| exists:', Path(SAMPLE_DIR).exists())
    print('VIDEO_SEARCH_DIRS:')
    for d in (VIDEO_SEARCH_DIRS or [])[:12]:
        print(' -', d, '| exists:', Path(d).exists())
    print('video count:', len(videos))
    for v in videos[:max_show]:
        print(' -', v)
    if not videos:
        print('\n[중요] 영상이 0개입니다.')
        print('네 실제 영상 폴더가 C:/Users/우성한/Sample (1)/Sample 라면 아래를 실행하세요:')
        print(r"set_video_root(r'C:/Users/우성한/Sample (1)/Sample')")
    return videos


def get_video_meta(video_path: Path) -> dict:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'영상 열기 실패: {video_path}')
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return {
        'width': width,
        'height': height,
        'fps': round(float(fps), 3) if fps else 0.0,
        'frame_count': frame_count,
        'length_sec': round(float(frame_count / fps), 3) if fps else 0.0,
    }


def read_frame(video_path: Path, frame_id: int):
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_id))
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError(f'프레임 읽기 실패: {video_path}, frame={frame_id}')
    return frame


def sample_frame_ids(frame_count: int, num_samples=10, margin_ratio=0.05):
    if frame_count <= 0:
        return []
    start = int(frame_count * margin_ratio)
    end = int(frame_count * (1.0 - margin_ratio))
    if end <= start:
        start, end = 0, max(0, frame_count - 1)
    if num_samples <= 1:
        return [max(0, min(frame_count - 1, (start + end) // 2))]
    return sorted(set(int(round(x)) for x in np.linspace(start, end, num_samples)))


def bbox_center(bbox):
    x1, y1, x2, y2 = bbox
    return [round((x1 + x2) / 2, 2), round((y1 + y2) / 2, 2)]


def bbox_bottom_center(bbox):
    x1, y1, x2, y2 = bbox
    return [round((x1 + x2) / 2, 2), round(y2, 2)]


def bbox_area(bbox):
    x1, y1, x2, y2 = bbox
    return max(0, x2 - x1) * max(0, y2 - y1)


def bbox_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    union = bbox_area(a) + bbox_area(b) - inter
    return inter / union if union > 0 else 0.0


def point_to_bbox_distance(point, bbox):
    x, y = float(point[0]), float(point[1])
    x1, y1, x2, y2 = [float(v) for v in bbox]
    dx = max(x1 - x, 0, x - x2)
    dy = max(y1 - y, 0, y - y2)
    return math.sqrt(dx * dx + dy * dy)


def euclidean(a, b):
    dx = float(a[0]) - float(b[0])
    dy = float(a[1]) - float(b[1])
    return math.sqrt(dx * dx + dy * dy)


def normalize(v):
    n = math.sqrt(float(v[0]) ** 2 + float(v[1]) ** 2)
    if n < 1e-6:
        return [0.0, 0.0]
    return [float(v[0]) / n, float(v[1]) / n]


def cosine_similarity(a, b):
    ax, ay = float(a[0]), float(a[1])
    bx, by = float(b[0]), float(b[1])
    an = math.sqrt(ax * ax + ay * ay)
    bn = math.sqrt(bx * bx + by * by)
    if an < 1e-6 or bn < 1e-6:
        return 0.0
    return (ax * bx + ay * by) / (an * bn)


def median_point(points):
    if not points:
        return [0.0, 0.0]
    xs = [float(p[0]) for p in points]
    ys = [float(p[1]) for p in points]
    return [round(float(np.median(xs)), 2), round(float(np.median(ys)), 2)]


def clamp(v, lo=0.0, hi=1.0):
    return max(lo, min(hi, float(v)))


def nearest_frame_item(frames, frame_id):
    if not frames:
        return None
    return min(frames, key=lambda f: abs(int(f.get('frame_id', 0)) - int(frame_id)))


def draw_text_lines(frame, lines, origin=(30, 45), color=(0, 0, 255), scale=0.8, thickness=2):
    x, y = origin
    for i, line in enumerate(lines):
        cv2.putText(frame, str(line), (x, y + i * 32), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness)
    return frame

# 처음 실행 시 진단 출력
print_video_diagnostics(max_show=10)


cwd: C:\Users\우성한
SAMPLE_DIR: C:\Users\우성한 | exists: True
VIDEO_SEARCH_DIRS:
 - C:\Users\우성한 | exists: True
 - C:\Users\우성한\Sample (1) | exists: True
 - C:\Users\우성한\Sample (1)\Sample | exists: True
 - C:\Users | exists: True
 - C:\Users\우성한\videos | exists: True
video count: 133
 - C:\Users\우성한\AppData\Local\Google\Chrome\User Data\Default\Extensions\ohahllgiabjaoigichmmfljhkcfikeof\4.2.9_0\web-accessible-resources\redirects\noopmp4.mp4
 - C:\Users\우성한\AppData\Local\Microsoft\Office\SolutionPackages\1aa9fc2759deb38db92573cf34bbd8ff\PackageResources\OfflineFiles\offline-video_3f4546fb35377b5a8473d8d07b18aedb5e52d1b93fb46b2a79132de04b9e046c.mp4
 - C:\Users\우성한\AppData\Local\Steam\htmlcache\Default\Extensions\efaidnbmnnnibpcajpcglclefindmkaj\26.5.2.3_0\browser\images\LocalizedSummaryFte\de\summary.mp4
 - C:\Users\우성한\AppData\Local\Steam\htmlcache\Default\Extensions\efaidnbmnnnibpcajpcglclefindmkaj\26.5.2.3_0\browser\images\LocalizedSummaryFte\de\summary_nudge_local_file_access.mp4
 - C:\

[WindowsPath('C:/Users/우성한/AppData/Local/Google/Chrome/User Data/Default/Extensions/ohahllgiabjaoigichmmfljhkcfikeof/4.2.9_0/web-accessible-resources/redirects/noopmp4.mp4'),
 WindowsPath('C:/Users/우성한/AppData/Local/Microsoft/Office/SolutionPackages/1aa9fc2759deb38db92573cf34bbd8ff/PackageResources/OfflineFiles/offline-video_3f4546fb35377b5a8473d8d07b18aedb5e52d1b93fb46b2a79132de04b9e046c.mp4'),
 WindowsPath('C:/Users/우성한/AppData/Local/Steam/htmlcache/Default/Extensions/efaidnbmnnnibpcajpcglclefindmkaj/26.5.2.3_0/browser/images/LocalizedSummaryFte/de/summary.mp4'),
 WindowsPath('C:/Users/우성한/AppData/Local/Steam/htmlcache/Default/Extensions/efaidnbmnnnibpcajpcglclefindmkaj/26.5.2.3_0/browser/images/LocalizedSummaryFte/de/summary_nudge_local_file_access.mp4'),
 WindowsPath('C:/Users/우성한/AppData/Local/Steam/htmlcache/Default/Extensions/efaidnbmnnnibpcajpcglclefindmkaj/26.5.2.3_0/browser/images/LocalizedSummaryFte/en_US/summary.mp4'),
 WindowsPath('C:/Users/우성한/AppData/Local/Steam/htmlcach

In [3]:
# ============================================================
# 2. 모델 로딩 유틸
# ============================================================
MODEL_CACHE = {}


def get_device():
    if torch is not None:
        try:
            if torch.cuda.is_available():
                print('[device] GPU:', torch.cuda.get_device_name(0))
                return 0
        except Exception as e:
            print('[device] CUDA 확인 실패:', e)
    print('[device] CPU')
    return 'cpu'


def require_yolo():
    if YOLO is None:
        raise RuntimeError('ultralytics가 설치되어 있지 않습니다. pip install ultralytics 후 다시 실행하세요.')


def load_yolo_model(model_path_or_name, cache_key=None):
    require_yolo()
    key = cache_key or str(model_path_or_name)
    if key in MODEL_CACHE:
        return MODEL_CACHE[key]
    model_path_or_name = str(model_path_or_name)
    print('[model] loading:', model_path_or_name)
    model = YOLO(model_path_or_name)
    MODEL_CACHE[key] = model
    return model


def load_yoloworld_model(prompts, model_path_or_name=YOLO_WORLD_MODEL, cache_key='yoloworld_scene'):
    require_yolo()
    key = f'{cache_key}:{str(model_path_or_name)}:{len(prompts)}'
    if key in MODEL_CACHE:
        return MODEL_CACHE[key]
    print('[yolo-world] loading:', model_path_or_name)
    model = YOLO(str(model_path_or_name))
    if not hasattr(model, 'set_classes'):
        raise RuntimeError('현재 ultralytics/YOLO-World 모델이 set_classes를 지원하지 않습니다. ultralytics 업데이트가 필요합니다.')
    model.set_classes(prompts)
    MODEL_CACHE[key] = model
    return model

DEVICE = get_device()


[device] GPU: NVIDIA GeForce RTX 3060


## 3. 1단계 — 사람 + 이륜 이동수단 tracking

기존 침입 파트는 사람만 추적했지만, 여기서는 **사람 + bicycle + motorcycle**을 함께 추적합니다.  
이 결과는 헬멧, 인도 주행, 싸움, 스토킹이 모두 공유하는 1차 공통 근거입니다.

In [4]:
# ============================================================
# 3. Base tracking: person + bicycle + motorcycle
# ============================================================
def run_base_tracking(
    video_name,
    force=False,
    conf=TRACK_CONF,
    imgsz=IMG_SIZE,
    tracker_yaml=TRACKER_YAML,
    classes=PERSON_AND_TWO_WHEEL_IDS,
):
    video_path = find_video(video_name)
    stem = safe_stem(video_path.name)
    output_path = TRACKING_DIR / f'{stem}_base_tracking.json'

    if output_path.exists() and not force:
        print('[tracking] existing 사용:', output_path)
        return output_path

    model = load_yolo_model(PERSON_MODEL, cache_key='person_base')
    meta = get_video_meta(video_path)
    print('[tracking] video:', video_path.name)
    print('[tracking] meta :', meta)

    results = model.track(
        source=str(video_path),
        tracker=tracker_yaml,
        persist=True,
        stream=True,
        classes=classes,
        conf=conf,
        imgsz=imgsz,
        device=DEVICE,
        verbose=False,
    )

    frames = []
    names = getattr(model, 'names', {}) or {}

    for frame_idx, result in enumerate(results):
        timestamp = frame_idx / meta['fps'] if meta['fps'] else 0.0
        frame_data = {
            'frame_id': frame_idx,
            'timestamp': round(float(timestamp), 3),
            'objects': [],
        }

        if result.boxes is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()
            clss = result.boxes.cls.cpu().numpy()
            ids = result.boxes.id.cpu().numpy() if result.boxes.id is not None else [None] * len(boxes)

            for det_idx, (box, c, cls_id, tid) in enumerate(zip(boxes, confs, clss, ids)):
                cls_id = int(cls_id)
                label = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else str(cls_id)
                x1, y1, x2, y2 = box.tolist()
                bbox = [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))]
                if bbox_area(bbox) < 50:
                    continue

                obj_type = 'human' if cls_id == COCO_PERSON else ('two_wheeler' if cls_id in TWO_WHEEL_COCO_IDS else label)
                obj_id = str(int(tid)) if tid is not None else f'det_{frame_idx}_{det_idx}'

                frame_data['objects'].append({
                    'obj_id': obj_id,
                    'class_id': cls_id,
                    'obj_label': obj_type,
                    'raw_label': label,
                    'confidence': round(float(c), 4),
                    'bbox': bbox,
                    'center': bbox_center(bbox),
                    'foot_point': bbox_bottom_center(bbox),
                    'area': int(bbox_area(bbox)),
                })

        frames.append(frame_data)
        if frame_idx % 100 == 0:
            print(f'[tracking] frame={frame_idx}')

    output = {
        'filename': video_path.name,
        'source_path': str(video_path),
        'video_meta': meta,
        'model': str(PERSON_MODEL),
        'classes': classes,
        'tracking_result': frames,
    }
    save_json(output, output_path)
    return output_path


def collect_tracks(tracking_data, obj_label=None, raw_labels=None, min_len=1):
    tracks = defaultdict(list)
    for frame in tracking_data.get('tracking_result', []):
        for obj in frame.get('objects', []):
            if obj_label is not None and obj.get('obj_label') != obj_label:
                continue
            if raw_labels is not None and obj.get('raw_label') not in raw_labels:
                continue
            tid = obj.get('obj_id')
            item = dict(obj)
            item['frame_id'] = frame['frame_id']
            item['timestamp'] = frame['timestamp']
            tracks[tid].append(item)
    return {tid: pts for tid, pts in tracks.items() if len(pts) >= min_len}


def summarize_track_points(points):
    if not points:
        return {}
    first = median_point([p['center'] for p in points[:max(1, len(points)//5)]])
    last = median_point([p['center'] for p in points[-max(1, len(points)//5):]])
    movement = euclidean(first, last)
    confs = [p.get('confidence') for p in points if p.get('confidence') is not None]
    return {
        'length': len(points),
        'first_frame': points[0]['frame_id'],
        'last_frame': points[-1]['frame_id'],
        'first_center': first,
        'last_center': last,
        'movement_px': round(float(movement), 2),
        'mean_confidence': round(float(np.mean(confs)), 4) if confs else 0.0,
    }


## 4. 2단계 — YOLO-World 장면/헬멧/도로-인도 후보 검출

헬멧과 킥보드류는 COCO 기본 class만으로 부족할 수 있으므로 YOLO-World open-vocabulary prompt를 같이 사용합니다.

In [5]:
# ============================================================
# 4. YOLO-World scene detection
# ============================================================
def get_extra_frames_from_tracking(tracking_path: Path, max_frames=12):
    if not Path(tracking_path).exists():
        return []
    data = load_json(tracking_path)
    tracks = collect_tracks(data, obj_label='human', min_len=3)
    two_tracks = collect_tracks(data, obj_label='two_wheeler', min_len=2)
    items = []
    for tid, pts in {**tracks, **two_tracks}.items():
        sm = summarize_track_points(pts)
        items.append((tid, pts, sm.get('movement_px', 0), sm.get('length', 0)))
    items = sorted(items, key=lambda x: (x[2], x[3]), reverse=True)[:5]
    out = []
    for _, pts, _, _ in items:
        n = len(pts)
        for idx in [0, n//4, n//2, 3*n//4, n-1]:
            if 0 <= idx < n:
                out.append(pts[idx]['frame_id'])
    return sorted(set(out))[:max_frames]


def detect_yoloworld_on_frame(model, frame, prompts, conf=WORLD_CONF, imgsz=IMG_SIZE):
    results = model.predict(source=frame, conf=conf, imgsz=imgsz, device=DEVICE, verbose=False)
    result = results[0]
    objects = []
    if result.boxes is None:
        return objects

    boxes = result.boxes.xyxy.cpu().numpy()
    confs = result.boxes.conf.cpu().numpy()
    clss = result.boxes.cls.cpu().numpy()

    for box, c, cls_id in zip(boxes, confs, clss):
        cls_id = int(cls_id)
        label = prompts[cls_id] if 0 <= cls_id < len(prompts) else f'class_{cls_id}'
        x1, y1, x2, y2 = box.tolist()
        bbox = [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))]
        w = max(0, bbox[2] - bbox[0])
        h = max(0, bbox[3] - bbox[1])
        if w < 8 or h < 8:
            continue
        objects.append({
            'label': label,
            'confidence': round(float(c), 4),
            'bbox': bbox,
            'center': bbox_center(bbox),
            'bottom_center': bbox_bottom_center(bbox),
            'area': int(w * h),
        })
    return objects


def run_scene_detection(
    video_name,
    tracking_path=None,
    force=False,
    sample_count=12,
    conf=WORLD_CONF,
    imgsz=IMG_SIZE,
    prompts=SCENE_PROMPTS,
):
    video_path = find_video(video_name)
    stem = safe_stem(video_path.name)
    output_path = SCENE_DIR / f'{stem}_scene_yoloworld.json'

    if output_path.exists() and not force:
        print('[scene] existing 사용:', output_path)
        return output_path

    meta = get_video_meta(video_path)
    frame_ids = sample_frame_ids(meta['frame_count'], sample_count)
    if tracking_path is not None:
        frame_ids.extend(get_extra_frames_from_tracking(tracking_path))
    frame_ids = sorted(set(int(f) for f in frame_ids if 0 <= int(f) < meta['frame_count']))

    model = load_yoloworld_model(prompts, YOLO_WORLD_MODEL, cache_key='scene_world')
    frames = []
    for frame_id in frame_ids:
        print(f'[scene] detect frame={frame_id}')
        frame = read_frame(video_path, frame_id)
        objects = detect_yoloworld_on_frame(model, frame, prompts, conf=conf, imgsz=imgsz)
        frames.append({'frame_id': frame_id, 'objects': objects})

    output = {
        'filename': video_path.name,
        'source_path': str(video_path),
        'video_meta': meta,
        'model': str(YOLO_WORLD_MODEL),
        'prompts': prompts,
        'frames': frames,
    }
    save_json(output, output_path)
    return output_path


## 5. 3단계 — 인도/도로 segmentation

정확한 인도 주행 판단은 객체 bbox만으로는 불안정합니다. 가능하면 Cityscapes 계열 SegFormer로 픽셀 수준의 `road/sidewalk`를 얻고, 실패하면 YOLO-World의 `sidewalk/road` bbox 후보를 fallback으로 사용합니다.

In [6]:
# ============================================================
# 5. Sidewalk segmentation: SegFormer optional + YOLO-World fallback
# ============================================================
SEG_CACHE = {}


def load_segformer_cityscapes():
    if 'segformer' in SEG_CACHE:
        return SEG_CACHE['segformer']
    try:
        from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
        model_id = 'nvidia/segformer-b5-finetuned-cityscapes-1024-1024' if DEVICE != 'cpu' else 'nvidia/segformer-b0-finetuned-cityscapes-1024-1024'
        print('[segformer] loading:', model_id)
        processor = SegformerImageProcessor.from_pretrained(model_id)
        model = SegformerForSemanticSegmentation.from_pretrained(model_id)
        if torch is not None and DEVICE != 'cpu':
            model = model.to('cuda')
        model.eval()
        id2label = getattr(model.config, 'id2label', {})
        label2id = {str(v).lower(): int(k) for k, v in id2label.items()}
        SEG_CACHE['segformer'] = (processor, model, label2id, model_id)
        return SEG_CACHE['segformer']
    except Exception as e:
        print('[segformer] 사용 불가. fallback 사용:', e)
        SEG_CACHE['segformer'] = None
        return None


def run_sidewalk_segmentation(
    video_name,
    scene_path=None,
    force=False,
    sample_count=14,
    use_segformer=USE_SEGFORMER_FOR_SIDEWALK,
):
    video_path = find_video(video_name)
    stem = safe_stem(video_path.name)
    output_path = SEG_DIR / f'{stem}_sidewalk_segmentation.json'

    if output_path.exists() and not force:
        print('[seg] existing 사용:', output_path)
        return output_path

    meta = get_video_meta(video_path)
    frame_ids = sample_frame_ids(meta['frame_count'], sample_count)

    # scene detection 프레임과 맞춰서 같은 프레임의 segmentation도 확보
    if scene_path is not None and Path(scene_path).exists():
        scene = load_json(scene_path)
        frame_ids.extend([f['frame_id'] for f in scene.get('frames', [])])
    frame_ids = sorted(set(int(f) for f in frame_ids if 0 <= int(f) < meta['frame_count']))

    segformer = load_segformer_cityscapes() if use_segformer else None
    frames = []
    available = segformer is not None

    if available:
        processor, model, label2id, model_id = segformer
        road_ids = [v for k, v in label2id.items() if 'road' == k or k.endswith('road')]
        sidewalk_ids = [v for k, v in label2id.items() if 'sidewalk' in k]
        print('[segformer] road_ids:', road_ids, 'sidewalk_ids:', sidewalk_ids)

        for frame_id in frame_ids:
            print(f'[seg] frame={frame_id}')
            frame = read_frame(video_path, frame_id)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            inputs = processor(images=rgb, return_tensors='pt')
            if torch is not None and DEVICE != 'cpu':
                inputs = {k: v.to('cuda') for k, v in inputs.items()}
            with torch.no_grad():
                outputs = model(**inputs)
            logits = outputs.logits
            up = torch.nn.functional.interpolate(
                logits,
                size=rgb.shape[:2],
                mode='bilinear',
                align_corners=False,
            )
            pred = up.argmax(dim=1)[0].detach().cpu().numpy().astype(np.uint8)
            road_mask = np.isin(pred, road_ids) if road_ids else np.zeros_like(pred, dtype=bool)
            sidewalk_mask = np.isin(pred, sidewalk_ids) if sidewalk_ids else np.zeros_like(pred, dtype=bool)

            # JSON이 너무 커지지 않게 mask 자체는 저장하지 않고 요약 + npy 저장 경로만 둔다.
            mask_path = SEG_DIR / f'{stem}_frame_{frame_id}_seg.npy'
            np.save(mask_path, pred)

            frames.append({
                'frame_id': frame_id,
                'method': 'segformer_cityscapes',
                'mask_path': str(mask_path),
                'road_ratio': round(float(road_mask.mean()), 4),
                'sidewalk_ratio': round(float(sidewalk_mask.mean()), 4),
                'road_ids': road_ids,
                'sidewalk_ids': sidewalk_ids,
            })
    else:
        for frame_id in frame_ids:
            frames.append({
                'frame_id': frame_id,
                'method': 'unavailable',
                'mask_path': None,
                'road_ratio': None,
                'sidewalk_ratio': None,
                'road_ids': [],
                'sidewalk_ids': [],
            })

    output = {
        'filename': video_path.name,
        'source_path': str(video_path),
        'video_meta': meta,
        'available': available,
        'frames': frames,
        'fallback_note': 'SegFormer가 unavailable이면 sidewalk_reasoning에서 YOLO-World sidewalk/road bbox를 사용합니다.',
    }
    save_json(output, output_path)
    return output_path


def get_seg_label_at_point(seg_data, frame_id, point, radius=12):
    # 가장 가까운 segmentation frame 사용
    frames = seg_data.get('frames', [])
    valid = [f for f in frames if f.get('mask_path')]
    if not valid:
        return {'label': 'unknown', 'sidewalk_ratio': 0.0, 'road_ratio': 0.0, 'nearest_frame': None}
    nearest = min(valid, key=lambda f: abs(int(f['frame_id']) - int(frame_id)))
    pred = np.load(nearest['mask_path'])
    h, w = pred.shape[:2]
    x, y = int(round(point[0])), int(round(point[1]))
    x1, x2 = max(0, x-radius), min(w, x+radius+1)
    y1, y2 = max(0, y-radius), min(h, y+radius+1)
    patch = pred[y1:y2, x1:x2]
    if patch.size == 0:
        return {'label': 'unknown', 'sidewalk_ratio': 0.0, 'road_ratio': 0.0, 'nearest_frame': nearest['frame_id']}
    sidewalk_ids = nearest.get('sidewalk_ids', [])
    road_ids = nearest.get('road_ids', [])
    sidewalk_ratio = float(np.isin(patch, sidewalk_ids).mean()) if sidewalk_ids else 0.0
    road_ratio = float(np.isin(patch, road_ids).mean()) if road_ids else 0.0
    label = 'sidewalk' if sidewalk_ratio >= max(0.35, road_ratio + 0.10) else ('road' if road_ratio >= 0.35 else 'unknown')
    return {
        'label': label,
        'sidewalk_ratio': round(sidewalk_ratio, 4),
        'road_ratio': round(road_ratio, 4),
        'nearest_frame': nearest['frame_id'],
    }


## 6. 헬멧 여부 추론

핵심은 단순히 `helmet` 객체가 보였는지가 아니라, **이륜차와 가까운 사람 = rider 후보**를 먼저 찾고, 그 사람의 상단 head-zone 안에서 헬멧/맨머리 증거를 봅니다.

In [7]:
# ============================================================
# 6. Helmet reasoning
# ============================================================
def scene_objects_near_frame(scene_data, frame_id, tolerance=8):
    frames = scene_data.get('frames', [])
    if not frames:
        return []
    nearest = min(frames, key=lambda f: abs(int(f['frame_id']) - int(frame_id)))
    if abs(int(nearest['frame_id']) - int(frame_id)) > tolerance:
        return []
    return nearest.get('objects', [])


def is_two_wheel_scene_obj(obj):
    label = str(obj.get('label', '')).lower()
    return label in TWO_WHEEL_LABELS or any(k in label for k in ['motor', 'scooter', 'bike', 'bicycle', 'kickboard'])


def is_helmet_obj(obj):
    label = str(obj.get('label', '')).lower()
    return label in HELMET_LABELS or 'helmet' in label


def is_bare_head_obj(obj):
    label = str(obj.get('label', '')).lower()
    return label in BARE_HEAD_LABELS or 'without helmet' in label or 'no helmet' in label


def person_head_zone(person_bbox, frame_w=None, frame_h=None):
    x1, y1, x2, y2 = person_bbox
    h = max(1, y2 - y1)
    zone = [x1, y1, x2, int(round(y1 + h * 0.42))]
    return expand_bbox(zone, ratio=0.12, frame_w=frame_w, frame_h=frame_h)


def match_rider_candidates(frame_objects, scene_objects, meta):
    humans = [o for o in frame_objects if o.get('obj_label') == 'human']
    tws_track = [o for o in frame_objects if o.get('obj_label') == 'two_wheeler']
    tws_scene = [o for o in scene_objects if is_two_wheel_scene_obj(o)]
    two_wheelers = tws_track + tws_scene
    riders = []
    for person in humans:
        pb = person['bbox']
        pfoot = person.get('foot_point', bbox_bottom_center(pb))
        best = None
        best_score = 0.0
        for tw in two_wheelers:
            tb = tw['bbox']
            dist = point_to_bbox_distance(pfoot, expand_bbox(tb, ratio=0.20, frame_w=meta['width'], frame_h=meta['height']))
            overlap = bbox_iou(pb, expand_bbox(tb, ratio=0.35, frame_w=meta['width'], frame_h=meta['height']))
            # foot가 이륜차 bbox 근처에 있고, 사람 bbox와 이륜 bbox가 어느 정도 겹치면 rider 가능성 증가
            scale = max(60, math.sqrt(max(1, bbox_area(tb))))
            dist_score = clamp01(1.0 - dist / scale)
            score = max(dist_score, overlap * 2.0)
            if score > best_score:
                best_score = score
                best = tw
        if best is not None and best_score >= 0.18:
            riders.append({'person': person, 'two_wheeler': best, 'rider_score': round(float(best_score), 4)})
    return riders


def run_helmet_reasoning(tracking_path, scene_path, output_path=None, force=False):
    tracking = load_json(tracking_path)
    scene = load_json(scene_path)
    video_name = tracking['filename']
    stem = safe_stem(video_name)
    output_path = Path(output_path) if output_path else PRED_DIR / f'{stem}_helmet_prediction.json'
    if output_path.exists() and not force:
        print('[helmet] existing 사용:', output_path)
        return output_path

    meta = tracking['video_meta']
    evidences = []
    checked_rider_frames = 0
    helmet_hit = 0
    bare_hit = 0
    unknown_hit = 0

    frame_map = frame_objects_by_id(tracking)
    scene_frame_ids = [f['frame_id'] for f in scene.get('frames', [])]
    candidate_frame_ids = sorted(set(scene_frame_ids + list(frame_map.keys())[::max(1, len(frame_map)//30 or 1)]))

    for frame_id in candidate_frame_ids:
        frame_objects = frame_map.get(frame_id, [])
        scene_objects = scene_objects_near_frame(scene, frame_id, tolerance=12)
        riders = match_rider_candidates(frame_objects, scene_objects, meta)
        if not riders:
            continue
        for r in riders:
            checked_rider_frames += 1
            person = r['person']
            head_zone = person_head_zone(person['bbox'], meta['width'], meta['height'])
            helmets = [o for o in scene_objects if is_helmet_obj(o) and point_in_bbox(o.get('center', bbox_center(o['bbox'])), head_zone, margin=15)]
            bare_heads = [o for o in scene_objects if is_bare_head_obj(o) and point_in_bbox(o.get('center', bbox_center(o['bbox'])), head_zone, margin=15)]
            if helmets:
                helmet_hit += 1
                status = 'helmet'
            elif bare_heads:
                bare_hit += 1
                status = 'no_helmet_signal'
            else:
                unknown_hit += 1
                status = 'helmet_not_observed'
            evidences.append({
                'frame_id': frame_id,
                'person_track_id': person.get('obj_id'),
                'person_bbox': person.get('bbox'),
                'two_wheeler_bbox': r['two_wheeler'].get('bbox'),
                'rider_score': r['rider_score'],
                'head_zone': head_zone,
                'helmet_objects': helmets[:3],
                'bare_head_objects': bare_heads[:3],
                'status': status,
            })

    if checked_rider_frames < 2:
        final = {
            'detected': False,
            'predicted_event_class': 'none',
            'event_subtype': 'helmet_unknown',
            'confidence': 0.0,
            'status': 'unknown',
            'reason': '이륜차 탑승자 후보가 충분히 잡히지 않았습니다.',
        }
    else:
        helmet_ratio = helmet_hit / checked_rider_frames
        bare_ratio = bare_hit / checked_rider_frames
        no_helmet_signal_ratio = (bare_hit + unknown_hit * 0.45) / checked_rider_frames
        detected = (helmet_ratio < 0.28 and no_helmet_signal_ratio >= 0.55) or bare_ratio >= 0.30
        confidence = 0.0
        if detected:
            confidence = 0.48 + no_helmet_signal_ratio * 0.32 + bare_ratio * 0.16 + min(0.08, checked_rider_frames / 80)
            confidence = min(0.90, round(float(confidence), 3))
        final = {
            'detected': bool(detected),
            'predicted_event_class': '헬멧 미착용' if detected else 'none',
            'event_subtype': 'no_helmet' if detected else 'helmet_or_unknown',
            'confidence': confidence,
            'status': 'no_helmet' if detected else ('helmet_observed' if helmet_ratio >= 0.45 else 'unknown'),
            'checked_rider_frames': checked_rider_frames,
            'helmet_hit': helmet_hit,
            'bare_head_hit': bare_hit,
            'unknown_hit': unknown_hit,
            'helmet_ratio': round(float(helmet_ratio), 4),
            'bare_ratio': round(float(bare_ratio), 4),
            'no_helmet_signal_ratio': round(float(no_helmet_signal_ratio), 4),
            'reason': 'rider 후보의 head-zone에서 helmet 증거보다 bare/no-helmet 또는 미관측 신호가 우세합니다.' if detected else '헬멧 미착용으로 볼 증거가 충분하지 않습니다.',
        }

    output = {
        'video': video_name,
        'tracking_path': str(tracking_path),
        'scene_path': str(scene_path),
        'final_prediction': final,
        'evidence_frames': evidences[:80],
    }
    save_json(output, output_path)
    print_dict_compact(final)
    return output_path


## 7. 인도 주행 이륜차 추론

이 파트는 `two_wheeler`의 bottom-center가 인도 픽셀 또는 sidewalk bbox 위에 반복적으로 놓이는지 봅니다. SegFormer가 잡히면 픽셀 기반, 아니면 YOLO-World 기반 fallback으로 판단합니다.

In [8]:
# ============================================================
# 7. Sidewalk riding reasoning
# ============================================================
def is_sidewalk_obj(obj):
    label = str(obj.get('label', '')).lower()
    return label in SIDEWALK_LABELS or 'sidewalk' in label or 'pavement' in label or 'walkway' in label


def is_road_obj(obj):
    label = str(obj.get('label', '')).lower()
    return label in ROAD_LABELS or label in {'street', 'lane'}


def get_scene_area_label(scene_data, frame_id, point):
    objs = scene_objects_near_frame(scene_data, frame_id, tolerance=14)
    sidewalk_hits = []
    road_hits = []
    for o in objs:
        if is_sidewalk_obj(o) and point_in_bbox(point, o['bbox'], margin=12):
            sidewalk_hits.append(o)
        if is_road_obj(o) and point_in_bbox(point, o['bbox'], margin=12):
            road_hits.append(o)
    if sidewalk_hits and not road_hits:
        return {'label': 'sidewalk', 'method': 'yoloworld_bbox', 'objects': sidewalk_hits[:3]}
    if road_hits and not sidewalk_hits:
        return {'label': 'road', 'method': 'yoloworld_bbox', 'objects': road_hits[:3]}
    if sidewalk_hits and road_hits:
        # confidence가 큰 쪽
        sw_conf = max(float(o.get('confidence', 0)) for o in sidewalk_hits)
        rd_conf = max(float(o.get('confidence', 0)) for o in road_hits)
        label = 'sidewalk' if sw_conf >= rd_conf else 'road'
        return {'label': label, 'method': 'yoloworld_bbox_mixed', 'objects': (sidewalk_hits if label == 'sidewalk' else road_hits)[:3]}
    return {'label': 'unknown', 'method': 'none', 'objects': []}


def run_sidewalk_reasoning(tracking_path, seg_path, scene_path, output_path=None, force=False):
    tracking = load_json(tracking_path)
    seg = load_json(seg_path)
    scene = load_json(scene_path)
    video_name = tracking['filename']
    stem = safe_stem(video_name)
    output_path = Path(output_path) if output_path else PRED_DIR / f'{stem}_sidewalk_riding_prediction.json'
    if output_path.exists() and not force:
        print('[sidewalk] existing 사용:', output_path)
        return output_path

    frame_map = frame_objects_by_id(tracking)
    two_tracks = collect_tracks(tracking, obj_label='two_wheeler', min_len=2)

    # YOLO tracking에서 이륜차가 약하면 scene detection의 이륜차 bbox도 검사한다.
    evidence = []
    total_checks = 0
    sidewalk_checks = 0
    road_checks = 0

    # 1) tracked two-wheeler 우선
    for tid, pts in two_tracks.items():
        step = max(1, len(pts) // 24)
        for item in pts[::step]:
            point = item.get('foot_point') or bbox_bottom_center(item['bbox'])
            seg_label = get_seg_label_at_point(seg, item['frame_id'], point, radius=14)
            if seg_label['label'] == 'unknown':
                scene_label = get_scene_area_label(scene, item['frame_id'], point)
            else:
                scene_label = {'label': seg_label['label'], 'method': 'segformer', 'objects': []}

            total_checks += 1
            label = scene_label['label']
            if label == 'sidewalk':
                sidewalk_checks += 1
            elif label == 'road':
                road_checks += 1
            evidence.append({
                'source': 'tracking_two_wheeler',
                'frame_id': item['frame_id'],
                'track_id': tid,
                'bbox': item['bbox'],
                'point': point,
                'area_label': label,
                'method': scene_label['method'],
                'seg_detail': seg_label,
                'confidence': item.get('confidence'),
            })

    # 2) tracking에서 못 잡힌 경우 scene detection으로 보완
    if total_checks < 3:
        for frame in scene.get('frames', []):
            for obj in frame.get('objects', []):
                if not is_two_wheel_scene_obj(obj):
                    continue
                point = obj.get('bottom_center') or bbox_bottom_center(obj['bbox'])
                seg_label = get_seg_label_at_point(seg, frame['frame_id'], point, radius=14)
                if seg_label['label'] == 'unknown':
                    scene_label = get_scene_area_label(scene, frame['frame_id'], point)
                else:
                    scene_label = {'label': seg_label['label'], 'method': 'segformer', 'objects': []}
                total_checks += 1
                label = scene_label['label']
                if label == 'sidewalk':
                    sidewalk_checks += 1
                elif label == 'road':
                    road_checks += 1
                evidence.append({
                    'source': 'scene_two_wheeler',
                    'frame_id': frame['frame_id'],
                    'track_id': None,
                    'label': obj.get('label'),
                    'bbox': obj['bbox'],
                    'point': point,
                    'area_label': label,
                    'method': scene_label['method'],
                    'seg_detail': seg_label,
                    'confidence': obj.get('confidence'),
                })

    if total_checks == 0:
        final = {
            'detected': False,
            'predicted_event_class': 'none',
            'event_subtype': 'sidewalk_riding_unknown',
            'confidence': 0.0,
            'reason': '이륜차 후보가 충분히 검출되지 않았습니다.',
        }
    else:
        sidewalk_ratio = sidewalk_checks / total_checks
        road_ratio = road_checks / total_checks
        detected = sidewalk_checks >= 2 and sidewalk_ratio >= 0.42 and sidewalk_ratio > road_ratio
        confidence = 0.0
        if detected:
            # SegFormer 기반 증거가 많으면 신뢰도를 조금 높인다.
            seg_based = sum(1 for e in evidence if e.get('area_label') == 'sidewalk' and e.get('method') == 'segformer')
            confidence = 0.52 + sidewalk_ratio * 0.32 + min(0.10, seg_based / max(1, sidewalk_checks) * 0.10)
            confidence = min(0.92, round(float(confidence), 3))
        final = {
            'detected': bool(detected),
            'predicted_event_class': '인도 주행' if detected else 'none',
            'event_subtype': 'two_wheeler_sidewalk_riding' if detected else 'not_sidewalk_riding',
            'confidence': confidence,
            'total_checks': total_checks,
            'sidewalk_checks': sidewalk_checks,
            'road_checks': road_checks,
            'sidewalk_ratio': round(float(sidewalk_ratio), 4),
            'road_ratio': round(float(road_ratio), 4),
            'reason': '이륜차 기준점이 여러 프레임에서 sidewalk 영역에 위치했습니다.' if detected else '인도 주행으로 보기에는 sidewalk 체류 비율이 부족합니다.',
        }

    output = {
        'video': video_name,
        'tracking_path': str(tracking_path),
        'segmentation_path': str(seg_path),
        'scene_path': str(scene_path),
        'final_prediction': final,
        'evidence_frames': evidence[:120],
    }
    save_json(output, output_path)
    print_dict_compact(final)
    return output_path


## 8. 싸움 추론

싸움은 객체 검출만으로 어렵기 때문에 `사람 2명 이상`, `bbox 근접/겹침`, `짧은 시간 내 큰 움직임`, `optical flow`를 조합합니다.  
정확도를 더 높이고 싶으면 여기서 YOLO11-pose keypoint 기반 손/팔 속도 조건을 추가하면 됩니다.

In [9]:
# ============================================================
# 8. Fight reasoning: proximity + motion + optical flow
# ============================================================
def object_at_frame_from_track(points, frame_id, max_gap=2):
    if not points:
        return None
    best = min(points, key=lambda p: abs(int(p['frame_id']) - int(frame_id)))
    if abs(int(best['frame_id']) - int(frame_id)) <= max_gap:
        return best
    return None


def compute_track_velocity(points_by_id, tid, frame_id, gap=5):
    pts = points_by_id.get(tid, [])
    cur = object_at_frame_from_track(pts, frame_id, max_gap=gap)
    prev = object_at_frame_from_track(pts, frame_id - gap, max_gap=gap)
    if cur is None or prev is None:
        return [0.0, 0.0]
    return [cur['center'][0] - prev['center'][0], cur['center'][1] - prev['center'][1]]


def flow_magnitude_in_bbox(prev_gray, gray, bbox):
    x1, y1, x2, y2 = [int(v) for v in bbox]
    h, w = gray.shape[:2]
    x1, x2 = max(0, x1), min(w, x2)
    y1, y2 = max(0, y1), min(h, y2)
    if x2 <= x1 or y2 <= y1 or (x2 - x1) * (y2 - y1) < 100:
        return 0.0
    p = prev_gray[y1:y2, x1:x2]
    g = gray[y1:y2, x1:x2]
    flow = cv2.calcOpticalFlowFarneback(p, g, None, 0.5, 2, 15, 2, 5, 1.1, 0)
    mag = np.sqrt(flow[..., 0] ** 2 + flow[..., 1] ** 2)
    return float(np.percentile(mag, 80))


def union_bbox(a, b, meta, ratio=0.18):
    x1 = min(a[0], b[0])
    y1 = min(a[1], b[1])
    x2 = max(a[2], b[2])
    y2 = max(a[3], b[3])
    return expand_bbox([x1, y1, x2, y2], ratio=ratio, frame_w=meta['width'], frame_h=meta['height'])


def run_fight_reasoning(
    video_name,
    tracking_path,
    output_path=None,
    force=False,
    frame_step=6,
    min_pair_frames=4,
):
    tracking = load_json(tracking_path)
    video_path = find_video(video_name)
    video_name = tracking['filename']
    stem = safe_stem(video_name)
    output_path = Path(output_path) if output_path else PRED_DIR / f'{stem}_fight_prediction.json'
    if output_path.exists() and not force:
        print('[fight] existing 사용:', output_path)
        return output_path

    meta = tracking['video_meta']
    humans_by_id = collect_tracks(tracking, obj_label='human', min_len=4)
    frame_map = frame_objects_by_id(tracking)
    frame_ids = sorted(frame_map.keys())
    if not frame_ids or len(humans_by_id) < 2:
        final = {
            'detected': False,
            'predicted_event_class': 'none',
            'event_subtype': 'fight',
            'confidence': 0.0,
            'reason': '싸움 판단에 필요한 2명 이상의 human track이 부족합니다.',
        }
        output = {'video': video_name, 'tracking_path': str(tracking_path), 'final_prediction': final, 'evidence_frames': []}
        save_json(output, output_path)
        return output_path

    cap = cv2.VideoCapture(str(video_path))
    prev_gray = None
    prev_frame_id = None
    evidence = []

    sampled_frame_ids = frame_ids[::max(1, frame_step)]
    for frame_id in sampled_frame_ids:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_id))
        ret, frame = cap.read()
        if not ret:
            continue
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        humans = [o for o in frame_map.get(frame_id, []) if o.get('obj_label') == 'human']
        if len(humans) < 2:
            prev_gray, prev_frame_id = gray, frame_id
            continue

        for i in range(len(humans)):
            for j in range(i + 1, len(humans)):
                a, b = humans[i], humans[j]
                ac, bc = a['center'], b['center']
                dist = euclidean(ac, bc)
                ah = max(1, a['bbox'][3] - a['bbox'][1])
                bh = max(1, b['bbox'][3] - b['bbox'][1])
                scale = max(50, (ah + bh) / 2)
                close_score = clamp01(1.0 - dist / (scale * 1.55))
                overlap_score = clamp01(bbox_iou(a['bbox'], b['bbox']) * 3.0)
                va = compute_track_velocity(humans_by_id, a['obj_id'], frame_id, gap=frame_step)
                vb = compute_track_velocity(humans_by_id, b['obj_id'], frame_id, gap=frame_step)
                rel_motion = euclidean(va, vb)
                motion_score = clamp01(rel_motion / 55.0)

                flow_score = 0.0
                flow_mag = 0.0
                if prev_gray is not None and prev_frame_id is not None and abs(frame_id - prev_frame_id) <= frame_step * 2:
                    ub = union_bbox(a['bbox'], b['bbox'], meta)
                    flow_mag = flow_magnitude_in_bbox(prev_gray, gray, ub)
                    flow_score = clamp01(flow_mag / 8.0)

                pair_score = (
                    close_score * 0.36
                    + overlap_score * 0.20
                    + motion_score * 0.24
                    + flow_score * 0.20
                )

                if pair_score >= 0.45:
                    evidence.append({
                        'frame_id': frame_id,
                        'track_a': a['obj_id'],
                        'track_b': b['obj_id'],
                        'bbox_a': a['bbox'],
                        'bbox_b': b['bbox'],
                        'distance_px': round(float(dist), 2),
                        'close_score': round(float(close_score), 4),
                        'overlap_score': round(float(overlap_score), 4),
                        'motion_score': round(float(motion_score), 4),
                        'flow_magnitude': round(float(flow_mag), 4),
                        'flow_score': round(float(flow_score), 4),
                        'pair_score': round(float(pair_score), 4),
                    })

        prev_gray, prev_frame_id = gray, frame_id

    cap.release()

    if not evidence:
        final = {
            'detected': False,
            'predicted_event_class': 'none',
            'event_subtype': 'fight',
            'confidence': 0.0,
            'reason': '근접/움직임/flow 조건을 동시에 만족하는 사람 쌍이 없습니다.',
        }
    else:
        # 같은 pair가 여러 프레임에서 반복되면 싸움 신호로 강화
        pair_counter = Counter(tuple(sorted([e['track_a'], e['track_b']])) for e in evidence)
        best_pair, best_count = pair_counter.most_common(1)[0]
        pair_evs = [e for e in evidence if tuple(sorted([e['track_a'], e['track_b']])) == best_pair]
        best_score = max(e['pair_score'] for e in pair_evs)
        mean_score = float(np.mean([e['pair_score'] for e in pair_evs]))
        duration_est = (max(e['frame_id'] for e in pair_evs) - min(e['frame_id'] for e in pair_evs)) / max(1e-6, meta['fps']) if len(pair_evs) >= 2 else 0.0
        detected = best_count >= min_pair_frames and mean_score >= 0.50
        confidence = 0.0
        if detected:
            confidence = 0.50 + mean_score * 0.28 + min(0.12, best_count / 20) + min(0.08, duration_est / 8)
            confidence = min(0.92, round(float(confidence), 3))
        final = {
            'detected': bool(detected),
            'predicted_event_class': '싸움' if detected else 'none',
            'event_subtype': 'fight',
            'confidence': confidence,
            'best_pair': list(best_pair),
            'best_pair_evidence_count': int(best_count),
            'best_pair_mean_score': round(mean_score, 4),
            'best_pair_max_score': round(float(best_score), 4),
            'duration_est_sec': round(float(duration_est), 3),
            'reason': '두 사람의 근접, 상대 움직임, optical flow가 여러 프레임에서 반복되었습니다.' if detected else '싸움 후보는 있으나 반복성과 평균 점수가 부족합니다.',
        }

    output = {
        'video': video_name,
        'tracking_path': str(tracking_path),
        'final_prediction': final,
        'evidence_frames': evidence[:120],
    }
    save_json(output, output_path)
    print_dict_compact(final)
    return output_path


## 9. 스토킹 추론

스토킹은 단일 프레임이 아니라 **시간적으로 지속되는 관계**가 중요합니다. 여기서는 두 사람 track이 긴 시간 동안 비슷한 방향으로 이동하고, 한 명이 일정 거리 뒤에서 따라가는지를 봅니다.

In [10]:
# ============================================================
# 9. Stalking reasoning: trajectory pair relation
# ============================================================
def track_item_dict(points):
    return {int(p['frame_id']): p for p in points}


def common_frame_items(points_a, points_b, max_gap=2):
    # 정확한 같은 frame 우선, 부족하면 가까운 frame을 허용
    da = track_item_dict(points_a)
    db = track_item_dict(points_b)
    commons = []
    for fa, ia in da.items():
        if fa in db:
            commons.append((fa, ia, db[fa]))
        else:
            near = min(db.keys(), key=lambda x: abs(x - fa)) if db else None
            if near is not None and abs(near - fa) <= max_gap:
                commons.append((fa, ia, db[near]))
    return sorted(commons, key=lambda x: x[0])


def global_track_vector(points):
    if len(points) < 2:
        return [0.0, 0.0]
    k = max(1, len(points) // 5)
    first = median_point([p['center'] for p in points[:k]])
    last = median_point([p['center'] for p in points[-k:]])
    return [last[0] - first[0], last[1] - first[1]]


def evaluate_follow_relation(leader_pts, follower_pts, fps, min_dist=45, max_dist=420):
    common = common_frame_items(leader_pts, follower_pts, max_gap=3)
    if len(common) < 8:
        return None
    leader_vec = normalize(global_track_vector(leader_pts))
    follower_vec = normalize(global_track_vector(follower_pts))
    dir_cos = cosine_similarity(leader_vec, follower_vec)
    if dir_cos < 0.35:
        return None

    follow_hits = []
    distances = []
    behind_values = []
    for frame_id, leader, follower in common:
        lc = leader['center']
        fc = follower['center']
        dist = euclidean(lc, fc)
        # leader 기준 이동방향 반대쪽에 follower가 있으면 behind 양수
        rel = [lc[0] - fc[0], lc[1] - fc[1]]
        behind = rel[0] * leader_vec[0] + rel[1] * leader_vec[1]
        is_follow = (min_dist <= dist <= max_dist) and (behind > 20)
        follow_hits.append(is_follow)
        distances.append(dist)
        behind_values.append(behind)

    ratio = sum(follow_hits) / len(follow_hits)
    duration = (common[-1][0] - common[0][0]) / max(1e-6, fps)
    mean_dist = float(np.mean(distances)) if distances else 0.0
    mean_behind = float(np.mean(behind_values)) if behind_values else 0.0
    score = clamp01(ratio * 0.55 + max(0, dir_cos) * 0.25 + min(1.0, duration / 8.0) * 0.20)
    return {
        'common_frames': len(common),
        'first_common_frame': common[0][0],
        'last_common_frame': common[-1][0],
        'duration_sec': round(float(duration), 3),
        'direction_cos': round(float(dir_cos), 4),
        'follow_ratio': round(float(ratio), 4),
        'mean_distance_px': round(mean_dist, 2),
        'mean_behind_projection': round(mean_behind, 2),
        'score': round(float(score), 4),
    }


def run_stalking_reasoning(tracking_path, output_path=None, force=False):
    tracking = load_json(tracking_path)
    video_name = tracking['filename']
    stem = safe_stem(video_name)
    output_path = Path(output_path) if output_path else PRED_DIR / f'{stem}_stalking_prediction.json'
    if output_path.exists() and not force:
        print('[stalking] existing 사용:', output_path)
        return output_path

    meta = tracking['video_meta']
    humans = collect_tracks(tracking, obj_label='human', min_len=10)
    candidates = []

    ids = list(humans.keys())
    for i in range(len(ids)):
        for j in range(len(ids)):
            if i == j:
                continue
            leader_id = ids[i]
            follower_id = ids[j]
            res = evaluate_follow_relation(humans[leader_id], humans[follower_id], meta['fps'])
            if res is None:
                continue
            res.update({'leader_track_id': leader_id, 'follower_track_id': follower_id})
            candidates.append(res)

    candidates = sorted(candidates, key=lambda x: x['score'], reverse=True)
    if not candidates:
        final = {
            'detected': False,
            'predicted_event_class': 'none',
            'event_subtype': 'stalking',
            'confidence': 0.0,
            'reason': '지속적인 뒤따름 관계를 만족하는 human track pair가 없습니다.',
        }
    else:
        best = candidates[0]
        detected = best['score'] >= 0.56 and best['duration_sec'] >= 2.0 and best['follow_ratio'] >= 0.42
        confidence = 0.0
        if detected:
            confidence = 0.48 + best['score'] * 0.35 + min(0.10, best['duration_sec'] / 20)
            confidence = min(0.90, round(float(confidence), 3))
        final = {
            'detected': bool(detected),
            'predicted_event_class': '스토킹' if detected else 'none',
            'event_subtype': 'stalking',
            'confidence': confidence,
            'best_candidate': best,
            'candidate_count': len(candidates),
            'reason': '한 사람이 다른 사람 뒤쪽에서 일정 거리와 방향성을 유지하며 따라가는 관계가 지속되었습니다.' if detected else '스토킹 후보는 있으나 지속 시간/비율/점수가 부족합니다.',
        }

    output = {
        'video': video_name,
        'tracking_path': str(tracking_path),
        'final_prediction': final,
        'candidates': candidates[:30],
    }
    save_json(output, output_path)
    print_dict_compact(final)
    return output_path


## 10. 선택 단계 — 기존 침입 파이프라인 연결

업로드된 `run_intrusion_detector.py`와 관련 파일이 같은 폴더에 있으면 그대로 호출합니다.  
이 노트북은 침입 코드를 새로 뜯어고치지 않고, 그 결과만 fusion 단계에 포함합니다.

In [11]:
# ============================================================
# 10. Optional intrusion pipeline adapter
# ============================================================
def run_optional_intrusion(video_name, output_path=None, force=False, mode='both'):
    video_path = find_video(video_name)
    stem = safe_stem(video_path.name)
    output_path = Path(output_path) if output_path else PRED_DIR / f'{stem}_intrusion_optional.json'
    if output_path.exists() and not force:
        print('[intrusion] existing 사용:', output_path)
        return output_path

    try:
        if str(BASE_DIR) not in sys.path:
            sys.path.append(str(BASE_DIR))
        from run_intrusion_detector import run_intrusion_detector
        result = run_intrusion_detector(video_path.name, mode=mode)
        output = {
            'video': video_path.name,
            'enabled': True,
            'mode': mode,
            'result': result,
            'final_prediction': {
                'detected': bool(result.get('detected_events')),
                'predicted_event_class': '비정상적인 경로로의 침입' if result.get('detected_events') else 'none',
                'event_subtype': 'intrusion',
                'confidence': max([float(e.get('confidence', 0.0)) for e in result.get('events', [])] + [0.0]),
                'reason': '기존 침입 파이프라인 결과를 통합했습니다.',
            }
        }
    except Exception as e:
        output = {
            'video': video_path.name,
            'enabled': False,
            'mode': mode,
            'error': str(e),
            'traceback': traceback.format_exc()[:3000],
            'final_prediction': {
                'detected': False,
                'predicted_event_class': 'none',
                'event_subtype': 'intrusion_unavailable',
                'confidence': 0.0,
                'reason': '침입 파이프라인 파일 또는 실행 환경이 준비되지 않았습니다.',
            }
        }
    save_json(output, output_path)
    print_dict_compact(output['final_prediction'])
    return output_path


## 11. Fusion — 전체 후보 이벤트 병합

각 파트가 독립적으로 `final_prediction`을 만들고, 마지막에 모두 합칩니다.  
영상 하나에 이상행동이 하나만 있다는 전제를 반영해 `primary_event`를 강하게 뽑되, 다른 후보도 함께 남깁니다.

In [12]:
# ============================================================
# 11. Fusion
# ============================================================
EVENT_WEIGHTS = {
    'two_wheeler_sidewalk_riding': 1.08,
    'no_helmet': 1.03,
    'fight': 1.12,
    'stalking': 1.06,
    'intrusion': 1.08,
    'gate_intrusion': 1.08,
    'fence_or_railing_intrusion': 1.08,
}

EVENT_PRIORITY = {
    'fight': 5,
    'stalking': 4,
    'two_wheeler_sidewalk_riding': 3,
    'no_helmet': 2,
    'intrusion': 3,
    'gate_intrusion': 3,
    'fence_or_railing_intrusion': 3,
}


def prediction_file_to_event(path, source_name):
    data = load_json(path)
    final = data.get('final_prediction', {})
    subtype = final.get('event_subtype') or final.get('intrusion_subtype') or source_name
    confidence = float(final.get('confidence', 0.0) or 0.0)
    detected = bool(final.get('detected', False))
    weight = EVENT_WEIGHTS.get(subtype, 1.0)
    # detected가 아니어도 후보 점수는 보관하되 primary 선정에서는 크게 불리하게 둔다.
    weighted_score = confidence * weight if detected else confidence * weight * 0.35
    return {
        'source': source_name,
        'path': str(path),
        'detected': detected,
        'event_class': final.get('predicted_event_class', 'none'),
        'event_subtype': subtype,
        'confidence': round(confidence, 4),
        'weighted_score': round(float(weighted_score), 4),
        'priority': EVENT_PRIORITY.get(subtype, 0),
        'reason': final.get('reason'),
        'summary': final,
    }


def fuse_predictions(video_name, prediction_paths, output_path=None):
    video_path = find_video(video_name)
    stem = safe_stem(video_path.name)
    output_path = Path(output_path) if output_path else FINAL_DIR / f'{stem}_bb_all_events.json'

    source_map = {
        'sidewalk': 'two_wheeler_sidewalk_riding',
        'helmet': 'no_helmet',
        'fight': 'fight',
        'stalking': 'stalking',
        'intrusion': 'intrusion',
    }

    events = []
    for source_name, path in prediction_paths.items():
        if path is None or not Path(path).exists():
            continue
        events.append(prediction_file_to_event(Path(path), source_map.get(source_name, source_name)))

    detected_events = [e for e in events if e.get('detected')]
    if detected_events:
        primary = sorted(detected_events, key=lambda e: (e['weighted_score'], e['priority'], e['confidence']), reverse=True)[0]
    elif events:
        primary = sorted(events, key=lambda e: (e['weighted_score'], e['priority'], e['confidence']), reverse=True)[0]
    else:
        primary = {
            'detected': False,
            'event_class': 'none',
            'event_subtype': 'none',
            'confidence': 0.0,
            'reason': '생성된 prediction이 없습니다.',
        }

    output = {
        'video': video_path.name,
        'detector': 'bb_all_in_one_abnormal_behavior',
        'final_primary_event': primary,
        'detected_events': sorted(detected_events, key=lambda e: e['weighted_score'], reverse=True),
        'all_candidate_events': sorted(events, key=lambda e: e['weighted_score'], reverse=True),
        'note': 'primary_event는 영상당 가장 의심도가 높은 1개 이벤트입니다. detected_events에는 복수 후보가 남습니다.',
    }
    save_json(output, output_path)

    print('\n================ BB FINAL ================')
    print('video   :', video_path.name)
    print('primary :', primary.get('event_class'), '/', primary.get('event_subtype'), '/', primary.get('confidence'))
    print('detected:', [(e['event_subtype'], e['confidence']) for e in output['detected_events']])
    print('saved   :', output_path)
    return output_path


## 12. 전체 실행 함수

아래 `run_bb_pipeline()` 하나만 호출하면, 한 영상에 대해 모든 파트를 순서대로 돌리고 최종 JSON을 만듭니다.

In [13]:
# ============================================================
# 12. All-in-one runner
# ============================================================
def run_bb_pipeline(
    video_name,
    force=False,
    run_intrusion=RUN_INTRUSION_BY_DEFAULT,
    scene_sample_count=12,
    seg_sample_count=14,
    fight_frame_step=6,
):
    t0 = time.time()
    video_path = find_video(video_name)
    print('\n########################################')
    print('[BB] video:', video_path.name)
    print('########################################')

    print('\n[1/8] base tracking')
    tracking_path = run_base_tracking(video_path.name, force=force)

    print('\n[2/8] YOLO-World scene detection')
    scene_path = run_scene_detection(video_path.name, tracking_path=tracking_path, force=force, sample_count=scene_sample_count)

    print('\n[3/8] sidewalk segmentation / fallback prepare')
    seg_path = run_sidewalk_segmentation(video_path.name, scene_path=scene_path, force=force, sample_count=seg_sample_count)

    print('\n[4/8] sidewalk riding reasoning')
    sidewalk_pred = run_sidewalk_reasoning(tracking_path, seg_path, scene_path, force=force)

    print('\n[5/8] helmet reasoning')
    helmet_pred = run_helmet_reasoning(tracking_path, scene_path, force=force)

    print('\n[6/8] fight reasoning')
    fight_pred = run_fight_reasoning(video_path.name, tracking_path, force=force, frame_step=fight_frame_step)

    print('\n[7/8] stalking reasoning')
    stalking_pred = run_stalking_reasoning(tracking_path, force=force)

    intrusion_pred = None
    if run_intrusion:
        print('\n[7.5/8] optional intrusion reasoning')
        intrusion_pred = run_optional_intrusion(video_path.name, force=force, mode='both')
    else:
        print('\n[7.5/8] optional intrusion reasoning skipped')

    print('\n[8/8] fusion')
    final_path = fuse_predictions(video_path.name, {
        'sidewalk': sidewalk_pred,
        'helmet': helmet_pred,
        'fight': fight_pred,
        'stalking': stalking_pred,
        'intrusion': intrusion_pred,
    })

    print(f'\n[BB done] elapsed={time.time() - t0:.1f}s')
    return final_path


def run_bb_batch(video_names=None, limit=None, force=False, run_intrusion=False):
    if video_names is None:
        videos = [p.name for p in list_videos()]
    else:
        videos = [str(v) for v in video_names]
    if limit is not None:
        videos = videos[:int(limit)]

    print('[batch] total:', len(videos))
    results = []
    for i, video_name in enumerate(videos, 1):
        print(f'\n\n========== [{i}/{len(videos)}] {video_name} ==========', flush=True)
        try:
            final_path = run_bb_pipeline(video_name, force=force, run_intrusion=run_intrusion)
            final = load_json(final_path)
            primary = final.get('final_primary_event', {})
            results.append({
                'video': video_name,
                'ok': True,
                'final_path': str(final_path),
                'primary_event': primary.get('event_class'),
                'subtype': primary.get('event_subtype'),
                'confidence': primary.get('confidence'),
                'detected_count': len(final.get('detected_events', [])),
            })
        except Exception as e:
            print('[batch error]', video_name, e)
            results.append({'video': video_name, 'ok': False, 'error': str(e)})

    out_path = FINAL_DIR / 'bb_batch_summary.json'
    save_json({'results': results}, out_path)
    print('\n[batch summary]', out_path)
    for r in results:
        print(r)
    return out_path



def run_first_bb_video(force=False, run_intrusion=False, **kwargs):
    """영상 목록의 첫 번째 파일로 파이프라인을 실행한다. 영상이 없으면 원인 안내를 먼저 출력한다."""
    videos = list_videos()
    print('[run_first] video count:', len(videos))
    if not videos:
        print_video_diagnostics(max_show=20)
        raise RuntimeError('실행할 영상이 없습니다. set_video_root(...)로 영상 폴더를 먼저 지정하세요.')
    return run_bb_pipeline(videos[0], force=force, run_intrusion=run_intrusion, **kwargs)


## 13. 실행 예시

아래 셀에서 원하는 영상명을 넣고 실행하세요.

- 한 영상만 실행: `run_bb_pipeline('파일명.mp4', run_intrusion=False)`
- 침입까지 포함: `run_bb_pipeline('파일명.mp4', run_intrusion=True)`
- 전체 배치: `run_bb_batch(run_intrusion=False)`

In [ ]:
# ============================================================
# 14. 실행 셀
# ============================================================
# 먼저 영상이 제대로 잡히는지 확인한다.
videos = print_video_diagnostics(max_show=20)

# 자동 탐색이 실패해서 video count가 0이면 아래 한 줄의 주석을 풀고 네 실제 영상 폴더로 바꿔 실행한다.
# set_video_root(r'C:/Users/우성한/Sample (1)/Sample')
# videos = print_video_diagnostics(max_show=20)

# 예시 1: 첫 번째 영상 1개 실행. videos[0]를 직접 쓰지 않아서 빈 리스트 오류가 나지 않는다.
# final_path = run_first_bb_video(force=False, run_intrusion=False)
# print('final:', final_path)

# 예시 2: 특정 영상명으로 실행
# final_path = run_bb_pipeline('bl_e0272_c1(2).mp4', force=False, run_intrusion=False)

# 예시 3: 침입까지 함께 실행. 침입 파이프라인 py 파일들이 같은 폴더에 있어야 함.
# final_path = run_first_bb_video(force=False, run_intrusion=True)

# 예시 4: 전체 영상 배치 실행
# summary_path = run_bb_batch(run_intrusion=False)


## 14. 결과 확인 셀

`outputs/bb/final/<영상명>_bb_all_events.json`에 최종 결과가 저장됩니다.

In [ ]:
def show_last_bb_results(limit=10):
    files = sorted(FINAL_DIR.glob('*_bb_all_events.json'), key=lambda p: p.stat().st_mtime, reverse=True)[:limit]
    print('result files:', len(files))
    for p in files:
        data = load_json(p)
        primary = data.get('final_primary_event', {})
        print('\n---', p.name)
        print('primary:', primary.get('event_class'), '/', primary.get('event_subtype'), '/', primary.get('confidence'))
        print('detected:', [(e.get('event_subtype'), e.get('confidence')) for e in data.get('detected_events', [])])

# show_last_bb_results()
